# Fine-Tuning BERT for Sentiment Analysis

Siguiendo los pasos del artículo de Bradney Smith [*A Complete Guide to BERT with Code*](https://towardsdatascience.com/a-complete-guide-to-bert-with-code-9f87602e4a11/).

> [!NOTE]
> Necesitas el dataset *IMDB Dataset of 50K Movie Reviews* (Kaggle) en `data/IMDB Dataset.csv`. Las salidas de este notebook se han limpiado: el entrenamiento completo necesita una GPU con ~12 GB de VRAM (ver la nota final).

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('data/IMDB Dataset.csv')
df.head(5)

## 1 Load and Preprocess the Dataset

### Cleaning the reviews

In [ ]:
print(f"Original:\n{df.iloc[1]['review'][0:72]}")

#Remove HTML break tags <br />
df['review_cleaned'] = df['review'].apply(lambda x: x.replace('<br />', ''))
print(f"With no break tags:\n{df.iloc[1]['review_cleaned'][0:72]}")

#Remove unnecesary whitespace
df['review_cleaned'] = df['review_cleaned'].replace(r'\s+', ' ', regex=True)
print(f"Cleaned:\n{df.iloc[1]['review_cleaned'][0:72]}")

### Encoding the sentiment

In [ ]:
df['sentiment_encoded'] = df['sentiment'].apply(lambda x:0 if x == 'negative' else 1)
df.head(5)

## 2 Tokenize the Data

In [ ]:
from transformers import BertTokenizer

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
tokenizer

### Visualization for a single review

In [ ]:
#Encode a sample sentence
sample = "I liked this movie"
#We can return token_ids tensor in pytorch, numpy or tensor flow format
token_ids = tokenizer.encode(sample, return_tensors='np')[0]
print(f"Token IDs:\n{token_ids}")

#Convert token_ids back to tokens. Just to visualize special tokens added
tokens = tokenizer.convert_ids_to_tokens(token_ids)
print(f"Tokens:\n{tokens}")

In [ ]:
review = df['review_cleaned'].iloc[0]
token_ids = tokenizer.encode(review, max_length = 512, padding = 'max_length', truncation = True, return_tensors = 'pt')
token_ids #Padding starts at 393th token

Nota: tokenizer.encode() puede devolver los token_ids tanto como un tensor de numpy, pytorch or tensorflow. Para este tipo de proyectos se recomienda el formato pytorch ya que esta mejor preparado para usarse en CUDA

**Nota:** en versiones antiguas de `transformers` se usaba `tokenizer.encode_plus(...)` para obtener a la vez los `input_ids` y la `attention_mask`. En la versión actual `encode_plus()` ya no existe (da `AttributeError`): lo mismo se hace llamando directamente a `tokenizer(...)`, como en la celda siguiente.

In [ ]:
review = df['review_cleaned'].iloc[0]
batch_encoder = tokenizer(review, max_length = 512, padding = 'max_length', truncation = True, return_tensors = 'pt')

In [ ]:
print(f"Batch encoder keys:\n{batch_encoder.keys()}\n")

In [ ]:
print(f"Attention mask:\n{batch_encoder['attention_mask']}")

### Encoding all reviews

In [ ]:
import torch

In [ ]:
%%time

token_ids = []
attention_masks = []

#Encode each review
for review in df['review_cleaned']:
    batch_encoder = tokenizer(review, max_length = 512, padding = 'max_length', truncation = True, return_tensors = 'pt')
    token_ids.append(batch_encoder['input_ids'])
    attention_masks.append(batch_encoder['attention_mask'])

#Convert token_ids and attention_masks to pytorch tensors
token_ids = torch.cat(token_ids, dim = 0)
attention_masks = torch.cat(attention_masks, dim = 0)

## 3 Create the Train and Validation DataLoaders

In [ ]:
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
val_size = 0.1

#Split the token IDs
train_ids, val_ids = train_test_split(token_ids, test_size=val_size, shuffle=False)

#Split the attention masks
train_masks, val_masks = train_test_split(attention_masks, test_size=val_size, shuffle=False)

#Split the labels
labels = torch.tensor(df['sentiment_encoded'].values)
train_labels, val_labels = train_test_split(labels, test_size=val_size, shuffle=False)

#Create the DataLoaders
train_data = TensorDataset(train_ids, train_masks, train_labels)
train_dataloader = DataLoader(train_data, shuffle = True, batch_size = 16)
val_data = TensorDataset(val_ids, val_masks, val_labels)
val_dataloader = DataLoader(val_data, batch_size = 16)

Nota: **Sobre suffle**
   - Al hacer las particiones es importante no mezclar los elementos (shuffle=False). De otro modo, perderiamos la asociaciones entre ids, mascaras y labels
   - En train_dataloader se tiene suffle=True porque el objetivo del conjunto de entrenamiento es que el modelo aprenda los patrones del texto, e ignore información asociada a las posiciones de los elementos
   - En data_loader hacer suffle es innecesario porque el modelo no obtiene información a partir de el, solo se usa para evaluar su rendimiento

## 4 Instantiate a BERT model

In [ ]:
from transformers import BertForSequenceClassification

In [ ]:
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels = 2)

Nota: Aunque parezca un error, este status report es normal. Significa que el modelo aun no ha sido entrenado para nuestra tarea, por lo que los pesos de la capa lineal aun no tienen valor definido

## 5 Instantitate an Optimizer, Loss Function and Scheduler

In [ ]:
from torch.optim import AdamW
import torch.nn as nn
from transformers import get_linear_schedule_with_warmup

In [ ]:
EPOCHS = 2

#Optimizer
#lr=2e-5: valor típico para fine-tuning de BERT. El valor por defecto de AdamW (1e-3) es demasiado alto
#y suele hacer que el modelo 'olvide' lo aprendido en el pre-entrenamiento
optimizer = AdamW(model.parameters(), lr = 2e-5)

#Loss function
loss_function = nn.CrossEntropyLoss()

#Scheduler: To gradually reduce the learning rate as the training process continues
num_training_steps = EPOCHS * len(train_dataloader)
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps = 0, num_training_steps = num_training_steps
)

## 6 Fine-Tunning Loop

Nota: Para tener mejores tiempos de entrenamiento podemos usar la potencia de la GPU a traves de la plataforma CUDA.
- Para poder utilizar CUDA es necesario instalar los drivers de NVIDIA

In [ ]:
#Check if GPUs with CUDA are available
if torch.cuda.is_available():
    print("CUDA available")
    device = torch.device('cuda:0')
else:
    print("CUDA not available")
    device = torch.device('cpu')

In [ ]:
import numpy as np

In [ ]:
def calculate_accuracy(preds, labels):
    pred_flat = np.argmax(preds, axis = 1).flatten()
    labels_flat = labels.flatten()
    accuracy = np.sum(pred_flat == labels_flat) / len(labels_flat)

    return accuracy

In [ ]:
model.to(device)

for epoch in range(0, EPOCHS):
    #1. Switch the model to be in "Train Mode". Activates the dropout layer
    model.train()

    #2. Save training loss to track it over subsequent epochs.
    #It should decrease with each epoch if training is successful
    training_loss = 0

    for batch in train_dataloader:

        #3. Move token IDs, attention masks and labels to GPU (if available)
        batch_token_ids = batch[0].to(device)
        batch_attention_mask = batch[1].to(device)
        batch_labels = batch[2].to(device)

        #4. Reset the calculated gradients from the previous iteration loop
        model.zero_grad()

        #5. Pass the batch to the model to calculate the logits (predictions based
        # on the current classifier parameters (weigths and biases)) and loss
        loss, logits = model (
            batch_token_ids,
            token_type_ids = None,
            attention_mask = batch_attention_mask,
            labels = batch_labels,
            return_dict = False
        )

        #6. Extract the total loss for the epoch
        #Loss is returned as a PyTorch tensor, .item() extracts its float value
        training_loss += loss.item()

        #7. Perform a backwar pass of the madel and propagate the loss through the classifier
        #head. This will allow the model to adjust its parameters to improve the performance
        loss.backward()

        #8. Clip the gradient to be no longer than 1.0.
        # So the model does not suffer from the exploding gradients problem
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        #9. Call the optimizer to take a step in the direction of the error surface as
        #determined by the backward pass
        optimizer.step()
        scheduler.step()

    #10. Calculate the average loss and time taken for training on the epoch
    average_train_loss = training_loss / len(train_dataloader)

    #Validation step for the epoch

    #11. Switch the model to "Evaluation Mode". Deactivates the dropout layer
    model.eval()

    #12. Set the validation loss to 0.
    val_loss = 0
    val_accuracy = 0

    #13. Split the validation data into batches (already done)

    for batch in val_dataloader:

        #14. Move token IDs, attention masks and labels to GPU (if available)
        batch_token_ids = batch[0].to(device)
        batch_attention_mask = batch[1].to(device)
        batch_labels = batch[2].to(device)

        #15. Invoke no_grad() to instruct the model not to calculate the gradients
        #since we will not be performing any optimization, only inference
        with torch.no_grad():
            #16. Pass the batch to the model to calculate the logits and the loss
            loss, logits = model (
                batch_token_ids,
                token_type_ids = None,
                attention_mask = batch_attention_mask,
                labels = batch_labels,
                return_dict = False
            )

        #17. Extract the logits and labels from the model and move them to the CPU
        logits = logits.detach().cpu().numpy()
        label_ids = batch_labels.to('cpu').numpy()

        #18. Increment the loss and calculate the accuracy based on the true labels
        #in the validation dataloader
        val_loss += loss.item()

        #19. Calculate the average loss and accuracy
        val_accuracy += calculate_accuracy(logits, label_ids)

    average_val_accuracy = val_accuracy / len(val_dataloader)


Nota: **Sobre CUDA y GPU**
Para poder realizar los entrenamientos, todos los elementos incluido el propio modelo deben estar en el mismo sitio, idealmente la GPU. Por ello es necesario incluir los llamadas a .to(device)

Nota:
Con los hiperparametros actuales, batch_size=16 en los DataLoaders y max_length=512 en Encoding, mi ordenador (RTX 3050 con 4GB VRAM) no es capaz de entrenar el modelo por falta de memoria. Se estima que se necesita alrededor de 12GB VRAM. Para poder completar el entrenamiento debo reducir los hiperparametros o entrenar un modelo más pequeño.